# Notebook 4: Peeling Apart the Variables

**Act II — Power** | **Duration:** 55 min | **Register:** Semi-Formal

---

Every statistics course tells you to "control for" confounders. But what does that *mean* geometrically? In this notebook you’ll see that "controlling for" a variable means **projecting it out** — peeling away the part explained by other predictors. The Frisch-Waugh-Lovell theorem makes this precise, and the geometry makes it obvious.

**Core theorem:** Frisch-Waugh-Lovell (FWL).

**What you’ll need:**
- `regression_geometry.core` — the projection engine
- `regression_geometry.plots` — static visualizations
- `regression_geometry.data` — the Meridian dataset

*"To see what one variable does, first remove what the others explain."*

In [ ]:
# ============================================================
# Notebook 04: "Peeling Apart the Variables"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from regression_geometry.core import (
    ColumnSpace, Projection, HatMatrix,
    frisch_waugh_lovell, demean
)
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

In [ ]:
# Load the Meridian dataset
mer = load_meridian()
y = mer["salary"].values
print(f"Meridian Analytics: {len(mer)} employees")
mer.head()

## Beat 1: The Multiple Regression Mystery

Start with a puzzle.

In [ ]:
# Simple regression: salary ~ experience
X_exp = mer[["experience"]].values
model_simple = sm.OLS(y, sm.add_constant(X_exp)).fit()
beta_exp_simple = model_simple.params[1]

# Multiple regression: salary ~ experience + education
X_exp_edu = mer[["experience", "education"]].values
model_multi = sm.OLS(y, sm.add_constant(X_exp_edu)).fit()
beta_exp_multi = model_multi.params[1]

print(f"Simple regression  (salary ~ experience):             β_experience = ${beta_exp_simple:,.2f}")
print(f"Multiple regression (salary ~ experience + education): β_experience = ${beta_exp_multi:,.2f}")
print()
print(f"The coefficient changed by ${beta_exp_multi - beta_exp_simple:,.2f} when you added education.")
print("Why?")

You were told "add controls to get the right answer." But what does "controlling for" actually **mean**? What is the coefficient **doing** differently when education is in the model? The geometry will make this precise.

### 🛑 PREDICT FIRST

When you add education to the salary ~ experience regression, β_experience changes. **In which direction? Will it increase or decrease? Why?**

Hint: experience and education are positively correlated. Education also has a positive effect on salary.

*Write your prediction before running the next cell.*

In [ ]:
# The correlation between experience and education
r_exp_edu = np.corrcoef(mer["experience"].values, mer["education"].values)[0, 1]
print(f"Correlation between experience and education: r = {r_exp_edu:.3f}")
print()
print(f"Simple β_experience  = ${beta_exp_simple:,.2f}")
print(f"Multiple β_experience = ${beta_exp_multi:,.2f}")
print()
print("Experience and education are positively correlated.")
print("In the simple regression, experience was getting 'credit' for some of")
print("education's effect on salary. Adding education as a control removes")
print("that borrowed credit.")

## Beat 2: The Geometry of Partialling Out

Here’s what FWL does in pictures. To isolate the effect of $x_2$ (education) while controlling for $x_1$ (experience):

1. **Residualize $y$ against $x_1$**: Remove the part of salary explained by experience alone. What’s left is "salary with experience removed."
2. **Residualize $x_2$ against $x_1$**: Remove the part of education explained by experience. What’s left is "education with experience removed" — the *unique* information in education.
3. **Regress** the residualized salary on the residualized education. The slope IS $\beta_2$ from the full regression.

Geometrically, you’re projecting everything onto the orthogonal complement of the experience subspace, then doing a simple regression in that cleaned-up space.

In [ ]:
# FWL decomposition for education (j=1, the second predictor column)
fwl_result = frisch_waugh_lovell(X_exp_edu, y, j=1)

print("Frisch-Waugh-Lovell decomposition:")
print(f"  β_education from full regression:   ${model_multi.params[2]:,.2f}")
print(f"  β from residualized regression:     ${fwl_result['beta_j']:,.2f}")
print(f"  Match? {np.isclose(model_multi.params[2], fwl_result['beta_j'])}")
print()
print("The coefficient is identical. FWL guarantees this.")

In [ ]:
# Three-panel FWL visualization
viz.plot_fwl_decomposition(X_exp_edu, y, j=1)

In [ ]:
# Also verify FWL for experience (j=0)
fwl_exp = frisch_waugh_lovell(X_exp_edu, y, j=0)

print(f"β_experience from full regression:   ${model_multi.params[1]:,.2f}")
print(f"β from FWL (residualized):           ${fwl_exp['beta_j']:,.2f}")
print(f"Match? {np.isclose(model_multi.params[1], fwl_exp['beta_j'])}")

## Beat 3: FWL Formally Stated

### Theorem Without Algebra

Partition the predictors as $X = [X_1 \;\; X_2]$. The FWL theorem says:

$$\hat{\beta}_2 = (X_2^\top M_1 X_2)^{-1} X_2^\top M_1 y$$

where $M_1 = I - X_1(X_1^\top X_1)^{-1}X_1^\top$ is the **annihilator** from Notebook 2 — it residualizes against $X_1$.

Translation: $M_1 y$ is "$y$ with $X_1$ removed" and $M_1 X_2$ is "$X_2$ with $X_1$ removed." The coefficient $\hat{\beta}_2$ is just the **simple regression** of the cleaned-up $y$ on the cleaned-up $X_2$.

One diagram. Two projections. One sentence: *the multiple regression coefficient is the simple regression coefficient in the residualized space.*

<details>
<summary><strong>Going Deeper: Formal proof that the coefficients are identical</strong></summary>

The full OLS normal equations are $X^\top X \hat{\beta} = X^\top y$. Partition:

$$\begin{bmatrix} X_1^\top X_1 & X_1^\top X_2 \\ X_2^\top X_1 & X_2^\top X_2 \end{bmatrix} \begin{bmatrix} \hat{\beta}_1 \\ \hat{\beta}_2 \end{bmatrix} = \begin{bmatrix} X_1^\top y \\ X_2^\top y \end{bmatrix}$$

From the second block row: $X_2^\top X_1 \hat{\beta}_1 + X_2^\top X_2 \hat{\beta}_2 = X_2^\top y$.

Substitute $\hat{\beta}_1$ from the first block row and simplify. After algebra, you get:

$$\hat{\beta}_2 = (X_2^\top M_1 X_2)^{-1} X_2^\top M_1 y$$

which is exactly OLS of $M_1 y$ on $M_1 X_2$ — the residualized regression.

</details>

## Beat 4: The Correlation Slider

What happens when the predictors are correlated? Let’s vary the correlation and see.

In [ ]:
# Demonstrate the effect of predictor correlation on FWL
np.random.seed(42)
n_sim = 200

print(f"{'Correlation r(x1,x2)':<25} {'β2 (full)':<14} {'SE(β2)':<12} {'‖M1x2‖/‖x2‖':<15}")
print("-" * 68)

for r in [0.0, 0.3, 0.6, 0.8, 0.95]:
    x1 = np.random.randn(n_sim)
    x2 = r * x1 + np.sqrt(1 - r**2) * np.random.randn(n_sim)
    y_sim = 2.0 * x1 + 3.0 * x2 + np.random.randn(n_sim)

    X_sim = np.column_stack([x1, x2])
    model_sim = sm.OLS(y_sim, sm.add_constant(X_sim)).fit()

    # Residualize x2 against x1
    cs_x1 = ColumnSpace(x1.reshape(-1, 1), add_intercept=True)
    x2_resid = cs_x1.residual(x2)
    ratio = np.linalg.norm(x2_resid) / np.linalg.norm(x2)

    print(f"r = {r:<22.2f} {model_sim.params[2]:<14.3f} {model_sim.bse[2]:<12.3f} {ratio:<15.3f}")

print()
print("As correlation increases:")
print("  - The residualized x2 shrinks (less unique information).")
print("  - The standard error of β2 explodes.")
print("  - The coefficient becomes unstable.")

When your predictors are correlated, "controlling for" one of them strips most of the information out of the other. The more correlated they are, the less "unique" information each contributes, and the less precisely you can estimate each coefficient.

This is the geometric root of **multicollinearity** — a preview of Notebook 6.

### 🛑 PREDICT FIRST

In the Meridian dataset, experience and education are correlated at r ≈ 0.4. When you add education to salary ~ experience:

1. Will the experience coefficient go **up**, **down**, or **stay the same**?
2. What about its **standard error** — will it increase or decrease?

*Write your predictions before running the next cell.*

In [ ]:
# Check your predictions on Meridian data
print("Simple regression (salary ~ experience):")
print(f"  β_experience = ${model_simple.params[1]:,.2f}, SE = ${model_simple.bse[1]:,.2f}")
print()
print("Multiple regression (salary ~ experience + education):")
print(f"  β_experience = ${model_multi.params[1]:,.2f}, SE = ${model_multi.bse[1]:,.2f}")
print()
if model_multi.params[1] < model_simple.params[1]:
    print("The coefficient decreased.")
else:
    print("The coefficient increased.")
if model_multi.bse[1] > model_simple.bse[1]:
    print("The standard error increased.")
else:
    print("The standard error decreased.")
print("Adding a correlated predictor changes what 'experience' means in the model.")

## Beat 5: Meridian — Peeling Gender

Now the substantive application. Let’s run the full model and use FWL to isolate the gender coefficient.

In [ ]:
# Full model: salary ~ experience + education + gender
X_full = mer[["experience", "education", "gender"]].values
cs_full = ColumnSpace(X_full, add_intercept=True)
proj_full = cs_full.project(y)

model_full = sm.OLS(y, sm.add_constant(X_full)).fit()

print("salary ~ experience + education + gender")
print(f"  β_experience = ${model_full.params[1]:>10,.2f}")
print(f"  β_education  = ${model_full.params[2]:>10,.2f}")
print(f"  β_gender     = ${model_full.params[3]:>10,.2f}")
print(f"  R²           = {model_full.rsquared:.4f}")

In [ ]:
# FWL: peel out experience and education to isolate the gender coefficient
# gender is column j=2 (0-indexed) in X_full
fwl_gender = frisch_waugh_lovell(X_full, y, j=2)

print(f"β_gender from full regression:    ${model_full.params[3]:,.2f}")
print(f"β_gender from FWL (residualized): ${fwl_gender['beta_j']:,.2f}")
print(f"Match? {np.isclose(model_full.params[3], fwl_gender['beta_j'])}")

In [ ]:
# Added variable plot for gender
viz.plot_added_variable(X_full, y, j=2)

### 🛑 DIAGNOSE FIRST

Look at the added variable plot above. **From the plot, estimate the gender coefficient.** Then check against the regression output.

The slope in the added variable plot IS the coefficient from the full model. That’s not a coincidence — it’s the FWL theorem.

*Write your estimate, then verify.*

In [ ]:
---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| FWL residualized vectors | `sm.OLS(x_j, sm.add_constant(X_others)).fit().resid` |
| Added variable plot | `statsmodels.graphics.regressionplots.plot_partregress` |
| Multiple regression coefficient | `model.params[j]` (= slope in residualized space) |

---

🔗 **Back to Statsmodels**

Statsmodels has tools for exactly these computations.

In [ ]:
# FWL manually with statsmodels
X_controls = mer[["experience", "education"]].values
y_resid_sm = sm.OLS(y, sm.add_constant(X_controls)).fit().resid
gender_resid_sm = sm.OLS(mer["gender"].values, sm.add_constant(X_controls)).fit().resid

# Simple regression of residualized y on residualized gender
model_fwl_sm = sm.OLS(y_resid_sm, sm.add_constant(gender_resid_sm)).fit()

print("Manual FWL via statsmodels:")
print(f"  β_gender (full model):      ${model_full.params[3]:,.2f}")
print(f"  β_gender (FWL manual):      ${model_fwl_sm.params[1]:,.2f}")
print(f"  Match? {np.isclose(model_full.params[3], model_fwl_sm.params[1], atol=0.01)}")
print()
print("statsmodels also provides added variable plots via:")
print("  statsmodels.graphics.regressionplots.plot_partregress")

✍️ **The Memo**

Write a memo (3 sentences) to Meridian’s Chief People Officer explaining why the estimated gender pay gap changed from the simple regression value to the multiple regression value when you added experience and education as controls.

**Forbidden words:** *Frisch-Waugh, residualize, orthogonal complement, annihilator.*

*Your memo here:*

...

## Geometric Scoreboard

Same 4 gauges as Notebook 3: θ, R², tr(H)/n, ‖e‖/‖y‖.

In [ ]:
---
### Summary

**What you learned:**

- **Controlling for a variable means residualizing against it** — projecting it out of both the response and the other predictors.
- **The multiple regression coefficient IS the simple regression coefficient in the residualized space.** This is the Frisch-Waugh-Lovell theorem.
- **When predictors are correlated, residualization strips away shared information**, making each coefficient less precisely estimated. This is the geometric root of multicollinearity.

**Key geometric insight:** ***"Controlling for" is not a statistical incantation — it is a geometric operation: projecting onto the orthogonal complement.***

### Next

You've been treating $y$ as a single vector. But each element $y_i$ contributes differently to the projection. Some observations pull the shadow harder than others. Notebook 5 asks: **who controls the shadow?**

---

## Summary and Bridge

**What you learned:**

- **Controlling for a variable means residualizing against it** — projecting it out of both the response and the other predictors.
- **The multiple regression coefficient IS the simple regression coefficient in the residualized space.** This is the Frisch-Waugh-Lovell theorem.
- **When predictors are correlated, residualization strips away shared information**, making each coefficient less precisely estimated. This is the geometric root of multicollinearity.

---

**Next:** You’ve been treating $y$ as a single vector. But each element $y_i$ contributes differently to the projection. Some observations pull the shadow harder than others. Notebook 5 asks: **who controls the shadow?**